# 🚀 TraderPro - Entraînement du Modèle de Prédiction BTC

Ce notebook entraîne un modèle LSTM pour prédire les cours OHLC (Open, High, Low, Close) du BTC pour le jour suivant.

**Instructions:**
1. Exécutez toutes les cellules dans l'ordre
2. À la fin, téléchargez les fichiers générés:
   - `btc_lstm_model.keras` (le modèle)
   - `btc_scaler.pkl` (le normaliseur)
   - `training_config.json` (configuration)
3. Placez ces fichiers dans `backend/app/ml/models/`

**Runtime recommandé:** GPU T4 (gratuit sur Colab)

## 1️⃣ Installation des dépendances

In [ ]:
!pip install yfinance pandas numpy scikit-learn tensorflow keras matplotlib --quiet
print("✅ Dépendances installées!")

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential, Model
from keras.layers import LSTM, Dense, Dropout, Input, BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras.optimizers import Adam

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

## 2️⃣ Configuration

In [ ]:
# ====== CONFIGURATION ======
# Modifiez ces paramètres selon vos besoins

CONFIG = {
    # Données
    "symbol": "BTC-USD",           # Symbole Yahoo Finance
    "years_of_data": 3,             # Années de données historiques
    "sequence_length": 60,          # Jours de contexte pour prédire (60 jours = ~2 mois)
    
    # Modèle LSTM
    "lstm_units_1": 128,            # Neurones première couche LSTM
    "lstm_units_2": 64,             # Neurones deuxième couche LSTM
    "dropout_rate": 0.2,            # Taux de dropout (régularisation)
    
    # Entraînement
    "batch_size": 32,
    "epochs": 100,                  # Max epochs (early stopping arrêtera avant)
    "learning_rate": 0.001,
    "validation_split": 0.15,       # 15% pour validation
    "test_split": 0.1,              # 10% pour test final
    
    # Outputs
    "target_columns": ["Open", "High", "Low", "Close"],  # Ce qu'on prédit
}

print("📊 Configuration:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")

## 3️⃣ Téléchargement des données historiques

In [ ]:
def download_btc_data(symbol: str, years: int) -> pd.DataFrame:
    """
    Télécharge les données historiques du BTC depuis Yahoo Finance.
    """
    end_date = datetime.now()
    start_date = end_date - timedelta(days=years * 365)
    
    print(f"📥 Téléchargement des données {symbol}...")
    print(f"   Période: {start_date.strftime('%Y-%m-%d')} → {end_date.strftime('%Y-%m-%d')}")
    
    df = yf.download(symbol, start=start_date, end=end_date, progress=False)
    
    # Nettoyer les données
    df = df.dropna()
    
    # S'assurer que l'index est datetime
    df.index = pd.to_datetime(df.index)
    
    print(f"✅ {len(df)} jours de données téléchargés")
    print(f"   Prix min: ${df['Low'].min():,.2f}")
    print(f"   Prix max: ${df['High'].max():,.2f}")
    print(f"   Prix actuel: ${df['Close'].iloc[-1]:,.2f}")
    
    return df

# Télécharger les données
df = download_btc_data(CONFIG["symbol"], CONFIG["years_of_data"])
df.head()

In [ ]:
# Visualiser les données
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['Close'], label='Prix de clôture', color='blue', linewidth=1)
plt.fill_between(df.index, df['Low'], df['High'], alpha=0.3, label='Range H-L')
plt.title(f'Historique BTC ({CONFIG["years_of_data"]} ans)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Prix (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4️⃣ Feature Engineering

In [ ]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crée des features techniques pour améliorer les prédictions.
    """
    df = df.copy()
    
    # Returns
    df['Returns'] = df['Close'].pct_change()
    df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # Moyennes mobiles
    for window in [7, 14, 30, 50]:
        df[f'SMA_{window}'] = df['Close'].rolling(window=window).mean()
        df[f'EMA_{window}'] = df['Close'].ewm(span=window, adjust=False).mean()
    
    # Volatilité
    df['Volatility_7'] = df['Returns'].rolling(window=7).std()
    df['Volatility_30'] = df['Returns'].rolling(window=30).std()
    
    # RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # MACD
    exp1 = df['Close'].ewm(span=12, adjust=False).mean()
    exp2 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = exp1 - exp2
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    # Bollinger Bands
    df['BB_Middle'] = df['Close'].rolling(window=20).mean()
    bb_std = df['Close'].rolling(window=20).std()
    df['BB_Upper'] = df['BB_Middle'] + (bb_std * 2)
    df['BB_Lower'] = df['BB_Middle'] - (bb_std * 2)
    df['BB_Position'] = (df['Close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])
    
    # ATR (Average True Range)
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['ATR'] = tr.rolling(window=14).mean()
    
    # Volume features
    df['Volume_SMA'] = df['Volume'].rolling(window=20).mean()
    df['Volume_Ratio'] = df['Volume'] / df['Volume_SMA']
    
    # Temporal features
    df['DayOfWeek'] = df.index.dayofweek
    df['Month'] = df.index.month
    df['Quarter'] = df.index.quarter
    
    # Range features
    df['Daily_Range'] = (df['High'] - df['Low']) / df['Close']
    df['Body'] = (df['Close'] - df['Open']) / df['Close']
    
    # Supprimer les NaN créés par les rolling windows
    df = df.dropna()
    
    print(f"✅ {len(df.columns)} features créées")
    print(f"   Données restantes après nettoyage: {len(df)} jours")
    
    return df

# Créer les features
df_features = create_features(df)
print(f"\nFeatures disponibles:")
print(df_features.columns.tolist())

## 5️⃣ Préparation des données pour LSTM

In [ ]:
def prepare_lstm_data(df: pd.DataFrame, config: dict):
    """
    Prépare les données au format LSTM:
    - Normalisation
    - Création des séquences
    - Split train/val/test
    """
    sequence_length = config["sequence_length"]
    target_columns = config["target_columns"]
    
    # Sélectionner les features pour l'entraînement
    # On utilise OHLCV + les features techniques
    feature_columns = [
        'Open', 'High', 'Low', 'Close', 'Volume',
        'Returns', 'SMA_7', 'SMA_14', 'SMA_30', 'EMA_7', 'EMA_14',
        'Volatility_7', 'RSI', 'MACD', 'MACD_Signal',
        'BB_Position', 'ATR', 'Volume_Ratio',
        'Daily_Range', 'Body'
    ]
    
    # Filtrer les colonnes existantes
    feature_columns = [col for col in feature_columns if col in df.columns]
    
    data = df[feature_columns].values
    targets = df[target_columns].values
    dates = df.index.values
    
    # Normalisation
    scaler_features = MinMaxScaler(feature_range=(0, 1))
    scaler_targets = MinMaxScaler(feature_range=(0, 1))
    
    data_scaled = scaler_features.fit_transform(data)
    targets_scaled = scaler_targets.fit_transform(targets)
    
    # Créer les séquences
    X, y, seq_dates = [], [], []
    for i in range(sequence_length, len(data_scaled)):
        X.append(data_scaled[i-sequence_length:i])
        y.append(targets_scaled[i])  # Prédire le jour suivant
        seq_dates.append(dates[i])
    
    X = np.array(X)
    y = np.array(y)
    seq_dates = np.array(seq_dates)
    
    # Split temporel (important: pas de shuffle pour les séries temporelles!)
    test_size = int(len(X) * config["test_split"])
    val_size = int(len(X) * config["validation_split"])
    train_size = len(X) - test_size - val_size
    
    X_train = X[:train_size]
    y_train = y[:train_size]
    
    X_val = X[train_size:train_size+val_size]
    y_val = y[train_size:train_size+val_size]
    
    X_test = X[train_size+val_size:]
    y_test = y[train_size+val_size:]
    test_dates = seq_dates[train_size+val_size:]
    
    print(f"📊 Données préparées:")
    print(f"   Features utilisées: {len(feature_columns)}")
    print(f"   Séquence: {sequence_length} jours → prédiction J+1")
    print(f"   Train: {len(X_train)} échantillons")
    print(f"   Validation: {len(X_val)} échantillons")
    print(f"   Test: {len(X_test)} échantillons")
    print(f"   Shape X: {X_train.shape}")
    print(f"   Shape y: {y_train.shape}")
    
    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'test_dates': test_dates,
        'scaler_features': scaler_features,
        'scaler_targets': scaler_targets,
        'feature_columns': feature_columns,
        'target_columns': target_columns
    }

# Préparer les données
data_dict = prepare_lstm_data(df_features, CONFIG)

## 6️⃣ Construction du modèle LSTM

In [ ]:
def build_lstm_model(input_shape: tuple, output_size: int, config: dict) -> Model:
    """
    Construit un modèle LSTM pour la prédiction OHLC.
    
    Architecture:
    - 2 couches LSTM empilées avec dropout
    - BatchNormalization pour stabiliser l'entraînement
    - Dense layers pour la sortie multi-output
    """
    model = Sequential([
        # Première couche LSTM
        LSTM(
            units=config["lstm_units_1"],
            return_sequences=True,
            input_shape=input_shape
        ),
        BatchNormalization(),
        Dropout(config["dropout_rate"]),
        
        # Deuxième couche LSTM
        LSTM(
            units=config["lstm_units_2"],
            return_sequences=False
        ),
        BatchNormalization(),
        Dropout(config["dropout_rate"]),
        
        # Couches Dense
        Dense(32, activation='relu'),
        Dropout(config["dropout_rate"] / 2),
        
        # Sortie: 4 valeurs (Open, High, Low, Close)
        Dense(output_size, activation='linear')
    ])
    
    # Compilation
    optimizer = Adam(learning_rate=config["learning_rate"])
    model.compile(
        optimizer=optimizer,
        loss='huber',  # Huber loss est robuste aux outliers
        metrics=['mae']
    )
    
    return model

# Construire le modèle
input_shape = (data_dict['X_train'].shape[1], data_dict['X_train'].shape[2])
output_size = len(CONFIG["target_columns"])

model = build_lstm_model(input_shape, output_size, CONFIG)
model.summary()

## 7️⃣ Entraînement

In [ ]:
# Callbacks pour optimiser l'entraînement
callbacks = [
    # Arrêt anticipé si pas d'amélioration
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    # Réduction du learning rate si plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
    # Sauvegarder le meilleur modèle
    ModelCheckpoint(
        'best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

print("🚀 Début de l'entraînement...")
print(f"   Epochs max: {CONFIG['epochs']}")
print(f"   Batch size: {CONFIG['batch_size']}")
print()

history = model.fit(
    data_dict['X_train'], data_dict['y_train'],
    validation_data=(data_dict['X_val'], data_dict['y_val']),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Entraînement terminé!")

In [ ]:
# Visualiser l'historique d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Loss pendant l\'entraînement')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_title('MAE pendant l\'entraînement')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8️⃣ Évaluation sur le jeu de test

In [ ]:
# Charger le meilleur modèle
model = keras.models.load_model('best_model.keras')

# Prédictions sur le test set
y_pred_scaled = model.predict(data_dict['X_test'], verbose=0)

# Dénormaliser
y_pred = data_dict['scaler_targets'].inverse_transform(y_pred_scaled)
y_true = data_dict['scaler_targets'].inverse_transform(data_dict['y_test'])

# Métriques par colonne
print("📊 Métriques sur le jeu de test:")
print("=" * 60)

metrics_results = {}
for i, col in enumerate(CONFIG["target_columns"]):
    mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
    r2 = r2_score(y_true[:, i], y_pred[:, i])
    mape = np.mean(np.abs((y_true[:, i] - y_pred[:, i]) / y_true[:, i])) * 100
    
    metrics_results[col] = {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape}
    
    print(f"\n{col}:")
    print(f"   MAE:  ${mae:,.2f}")
    print(f"   RMSE: ${rmse:,.2f}")
    print(f"   MAPE: {mape:.2f}%")
    print(f"   R²:   {r2:.4f}")

In [ ]:
# Visualiser les prédictions vs réalité (derniers 60 jours)
n_show = 60

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(CONFIG["target_columns"]):
    ax = axes[i]
    ax.plot(range(n_show), y_true[-n_show:, i], label='Réel', linewidth=2)
    ax.plot(range(n_show), y_pred[-n_show:, i], label='Prédit', linewidth=2, alpha=0.8)
    ax.set_title(f'{col} - Prédiction vs Réalité')
    ax.set_xlabel('Jours')
    ax.set_ylabel('Prix ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyse de la précision directionnelle (important pour le trading!)
def directional_accuracy(y_true, y_pred):
    """Calcule le % de fois où la direction est correctement prédite."""
    true_direction = np.diff(y_true) > 0
    pred_direction = np.diff(y_pred) > 0
    return np.mean(true_direction == pred_direction) * 100

print("\n🎯 Précision directionnelle (hausse/baisse):")
print("=" * 60)
for i, col in enumerate(CONFIG["target_columns"]):
    dir_acc = directional_accuracy(y_true[:, i], y_pred[:, i])
    print(f"   {col}: {dir_acc:.1f}%")
    metrics_results[col]['directional_accuracy'] = dir_acc

## 9️⃣ Export du modèle et configuration

In [ ]:
# Sauvegarder le modèle final
model.save('btc_lstm_model.keras')
print("✅ Modèle sauvegardé: btc_lstm_model.keras")

# Sauvegarder les scalers
scalers = {
    'scaler_features': data_dict['scaler_features'],
    'scaler_targets': data_dict['scaler_targets']
}
joblib.dump(scalers, 'btc_scaler.pkl')
print("✅ Scalers sauvegardés: btc_scaler.pkl")

# Configuration pour le backend
training_config = {
    'model_version': '1.0.0',
    'trained_at': datetime.now().isoformat(),
    'symbol': CONFIG['symbol'],
    'sequence_length': CONFIG['sequence_length'],
    'feature_columns': data_dict['feature_columns'],
    'target_columns': data_dict['target_columns'],
    'metrics': metrics_results,
    'training_samples': len(data_dict['X_train']),
    'epochs_trained': len(history.history['loss']),
    'final_loss': float(history.history['val_loss'][-1]),
}

with open('training_config.json', 'w') as f:
    json.dump(training_config, f, indent=2, default=str)
print("✅ Configuration sauvegardée: training_config.json")

print("\n" + "="*60)
print("📦 FICHIERS À TÉLÉCHARGER:")
print("="*60)
print("1. btc_lstm_model.keras  → Le modèle entraîné")
print("2. btc_scaler.pkl        → Les normaliseurs")
print("3. training_config.json  → Configuration & métriques")
print("\nPlacez ces fichiers dans: backend/app/ml/models/")

In [ ]:
# Créer un ZIP pour faciliter le téléchargement
import shutil

files_to_zip = ['btc_lstm_model.keras', 'btc_scaler.pkl', 'training_config.json']
shutil.make_archive('traderpro_model', 'zip', '.', '.')

# Ou télécharger individuellement via Colab
try:
    from google.colab import files
    print("\n📥 Téléchargement automatique...")
    for f in files_to_zip:
        files.download(f)
except:
    print("\n💡 Pour télécharger manuellement:")
    print("   - Cliquez sur l'icône dossier à gauche")
    print("   - Faites clic droit sur chaque fichier → Télécharger")

## 🔟 Test de prédiction (simulation)

In [ ]:
def predict_tomorrow(model, df_features, scalers, config):
    """
    Simule une prédiction pour demain en utilisant les dernières données.
    C'est exactement ce que fera le backend.
    """
    feature_columns = [
        'Open', 'High', 'Low', 'Close', 'Volume',
        'Returns', 'SMA_7', 'SMA_14', 'SMA_30', 'EMA_7', 'EMA_14',
        'Volatility_7', 'RSI', 'MACD', 'MACD_Signal',
        'BB_Position', 'ATR', 'Volume_Ratio',
        'Daily_Range', 'Body'
    ]
    feature_columns = [col for col in feature_columns if col in df_features.columns]
    
    # Prendre les dernières 60 jours
    sequence_length = config['sequence_length']
    last_data = df_features[feature_columns].tail(sequence_length).values
    
    # Normaliser
    last_data_scaled = scalers['scaler_features'].transform(last_data)
    
    # Reshape pour LSTM: (1, sequence_length, features)
    X = last_data_scaled.reshape(1, sequence_length, -1)
    
    # Prédire
    pred_scaled = model.predict(X, verbose=0)
    
    # Dénormaliser
    pred = scalers['scaler_targets'].inverse_transform(pred_scaled)[0]
    
    return {
        'open': float(pred[0]),
        'high': float(pred[1]),
        'low': float(pred[2]),
        'close': float(pred[3])
    }

# Test de prédiction
prediction = predict_tomorrow(model, df_features, scalers, CONFIG)

current_price = df_features['Close'].iloc[-1]
predicted_close = prediction['close']
change_pct = ((predicted_close - current_price) / current_price) * 100

print("\n" + "="*60)
print("🔮 PRÉDICTION POUR DEMAIN")
print("="*60)
print(f"\nPrix actuel: ${current_price:,.2f}")
print(f"\nPrédictions pour demain:")
print(f"   Open:  ${prediction['open']:,.2f}")
print(f"   High:  ${prediction['high']:,.2f}")
print(f"   Low:   ${prediction['low']:,.2f}")
print(f"   Close: ${prediction['close']:,.2f}")
print(f"\nVariation attendue: {change_pct:+.2f}%")
print(f"Signal: {'📈 HAUSSIER' if change_pct > 0.5 else '📉 BAISSIER' if change_pct < -0.5 else '➡️ NEUTRE'}")

---

## ✅ Entraînement terminé!

### Prochaines étapes:

1. **Téléchargez les 3 fichiers** générés
2. **Placez-les dans** `backend/app/ml/models/`
3. **Lancez le backend** - le service de prédiction sera automatiquement disponible
4. **Testez** via `GET /api/prediction/tomorrow`

### Pour ré-entraîner le modèle:
- Exécutez ce notebook une fois par mois
- Téléchargez les nouveaux fichiers
- Remplacez les anciens dans le backend

---

*TraderPro - AI Trading Assistant*